In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os
import json
import re
import sys
from collections import defaultdict

project_root = os.path.join(os.getcwd(), "..")
result_dir = os.path.join(project_root, "results")

date = '20260316'
model_size = '12'

dirname_pattern = f"gemma-3-{model_size}B-lr0.01-{date}_seed(\\d+)_target_concepts_initvecwith(.*)"


## 基本の結果読み込み

In [57]:
# 以下の訓練日とモデルサイズに該当する結果を全て読み込む．(実行した全てのseedとinitVecType。ただしまだtest計算が済んでいない場合はlogit_score_summary.jsonがないため読み込まない)

initVecType_to_results = defaultdict(dict)
for dirname in os.listdir(result_dir):
    if re.match(dirname_pattern, dirname):
        match_groups = re.search(dirname_pattern, dirname)
        seed = int(match_groups.group(1))
        initVecType = match_groups.group(2)
        if not os.path.exists(os.path.join(result_dir, dirname, 'logit_score_summary.json')):
            print(f"Warning: logit_score_summary.json not found in {dirname}. Skipping.")
            continue
        with open(os.path.join(result_dir, dirname, 'logit_score_summary.json'), "r") as f:
            result = json.load(f)
        initVecType_to_results[initVecType][seed] = result
initVecType_to_results

defaultdict(dict,
            {'category_COG': {0: {'Breakthrough': {'0': {'accuracy': 0.3333333333333333,
                 'precision': 0.33902323376007587,
                 'recall': 0.3333333333333333,
                 'F1': 0.3286274509803922},
                '1': {'accuracy': 0.3172043010752688,
                 'precision': 0.31566177659002664,
                 'recall': 0.3172043010752688,
                 'F1': 0.30080906148867315},
                '2': {'accuracy': 0.42473118279569894,
                 'precision': 0.4305174840099509,
                 'recall': 0.42473118279569894,
                 'F1': 0.41501574266453967},
                '3': {'accuracy': 0.2849462365591398,
                 'precision': 0.28846153846153844,
                 'recall': 0.2849462365591398,
                 'F1': 0.2836988148463558},
                '4': {'accuracy': 0.3548387096774194,
                 'precision': 0.3600501135384856,
                 'recall': 0.3548387096774194,
         

## epoch毎のlossの推移

In [50]:
# 以下の訓練日とモデルサイズに該当する結果を全て読み込む．(実行した全てのseedとinitVecType。ただしまだtest計算が済んでいない場合はlogit_score_summary.jsonがないため読み込まない)


epoch = 3

def load_file_and_extract_accs_at_epoch(date, model_size, result_dir, epoch):
    dirname_pattern = f"gemma-3-{model_size}B-lr0.01-{date}_seed(\\d+)_target_concepts_initvecwith(.*)"

    initVecType_to_seed_to_accs = defaultdict(dict)
    for dirname in os.listdir(result_dir):
        if re.match(dirname_pattern, dirname):
            match_groups = re.search(dirname_pattern, dirname)
            seed = int(match_groups.group(1))
            initVecType = match_groups.group(2)
            if not os.path.exists(os.path.join(result_dir, dirname, 'logit_score_summary.json')):
                # print(f"Warning: logit_score_summary.json not found in {dirname}. Skipping.")
                continue
            with open(os.path.join(result_dir, dirname, 'logit_score_summary.json'), "r") as f:
                concept_to_epoch_to_scores = json.load(f)
            initVecType_to_seed_to_accs[initVecType][seed] = [epoch_to_scores[str(epoch)]['accuracy'] for epoch_to_scores in concept_to_epoch_to_scores.values()]
    return initVecType_to_seed_to_accs

initVecType_to_seed_to_accs = load_file_and_extract_accs_at_epoch(date, model_size, result_dir, epoch)
initVecType_to_seed_to_accs

defaultdict(dict,
            {'zero_2': {0: [0.2903225806451613,
               0.3237410071942446,
               0.5298507462686567,
               0.2222222222222222,
               0.3532608695652174,
               0.37662337662337664,
               0.7785714285714286,
               0.4365079365079365,
               0.32786885245901637,
               0.4666666666666667,
               0.44680851063829785,
               0.5338345864661654,
               0.477124183006536,
               0.5855855855855856,
               0.3424657534246575,
               0.35384615384615387,
               0.3626373626373626,
               0.4603174603174603,
               0.4024390243902439,
               0.19444444444444445]},
             'category_COG_5': {0: [0.3978494623655914,
               0.539568345323741,
               0.3358208955223881,
               0.2777777777777778,
               0.30978260869565216,
               0.4155844155844156,
               0.828571428571428

In [41]:

epoch = 3
# epoch +=1
initVecType_to_seed_to_accs = load_file_and_extract_accs_at_epoch(date, model_size, result_dir, epoch)

for initVecType, seed_to_accs in initVecType_to_seed_to_accs.items():
    for seed, accs in seed_to_accs.items():
        mean_acc = np.mean(accs, axis=0)
        std_acc = np.std(accs, axis=0)
        print(f"InitVecType: {initVecType:15}, Seed: {seed}, Mean Accuracy at Epoch {epoch}: {mean_acc:.4f}, Std: {std_acc:.4f}")
        # plt.plot(accs, label=f"{initVecType} (seed={seed})")
    # accs = list(seed_to_accs.values())
    # mean_accs = np.mean(accs, axis=0)
    # std_accs = np.std(accs, axis=0)
    # plt.plot(mean_accs, label=initVecType)
    # plt.fill_between(range(len(mean_accs)), mean_accs - std_accs, mean_accs + std_accs, alpha=0.2)

InitVecType: category_COG   , Seed: 0, Mean Accuracy at Epoch 3: 0.4050, Std: 0.1306
InitVecType: other_category_COG, Seed: 0, Mean Accuracy at Epoch 3: 0.4300, Std: 0.1385


### 最高到達accuracyを取得する


In [58]:

def load_file_and_extract_max_accs(date, model_size, result_dir):
    """各initVecTypeとseedに対して、回したepochs中の最高到達accuracyを抽出する関数"""
    dirname_pattern = f"gemma-3-{model_size}B-lr0.01-{date}_seed(\\d+)_target_concepts_initvecwith(.*)"

    initVecType_to_seed_to_maxacc = defaultdict(dict)
    initVecType_to_seed_to_epoch_at_maxacc = defaultdict(dict)
    for dirname in os.listdir(result_dir):
        if re.match(dirname_pattern, dirname):
            match_groups = re.search(dirname_pattern, dirname)
            seed = int(match_groups.group(1))
            initVecType = match_groups.group(2)
            if not os.path.exists(os.path.join(result_dir, dirname, 'logit_score_summary.json')):
                # print(f"Warning: logit_score_summary.json not found in {dirname}. Skipping.")
                continue
            with open(os.path.join(result_dir, dirname, 'logit_score_summary.json'), "r") as f:
                concept_to_epoch_to_scores = json.load(f)

            # 各conceptについて、回したepochs中の最高到達accuracyとそのepochを抽出
            max_accs = []
            epochs_at_maxacc = []
            for epoch_to_scores in concept_to_epoch_to_scores.values():
                max_accuracy = 0
                max_epoch_at_maxacc = 0
                for epoch, scores in epoch_to_scores.items():
                    # print(f"Epoch: {epoch}, Accuracy: {scores['accuracy']}")
                    if scores['accuracy'] > max_accuracy:
                        max_accuracy = scores['accuracy']
                        max_epoch_at_maxacc = epoch
                max_accs.append(max_accuracy)
                epochs_at_maxacc.append(max_epoch_at_maxacc)
            # print(f"Max Accuracies: {max_accs}")
            # print(f"Epochs at Max Accuracies: {epochs_at_maxacc}")
            initVecType_to_seed_to_maxacc[initVecType][seed] = max_accs
            initVecType_to_seed_to_epoch_at_maxacc[initVecType][seed] = epochs_at_maxacc
    return initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc

# 訓練日とモデルサイズに該当する結果を全て読み込む．(実行した全てのseedとinitVecType。ただしまだtest計算が済んでいない場合はlogit_score_summary.jsonがないため読み込まない)
initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(date, model_size, result_dir)

In [59]:

initVecType_to_seed_to_maxacc, initVecType_to_seed_to_epoch_at_maxacc = load_file_and_extract_max_accs(date, model_size, result_dir)

for initVecType, seed_to_maxacc in initVecType_to_seed_to_maxacc.items():
    for seed, max_accs in seed_to_maxacc.items():
        mean_max_acc = np.mean(max_accs)
        std_max_acc = np.std(max_accs)
        print(f"InitVecType: {initVecType:15}, Seed: {seed}, Mean Max Accuracy: {mean_max_acc:.4f}, Std: {std_max_acc:.4f}")

for initVecType, seed_to_epoch_at_maxacc in initVecType_to_seed_to_epoch_at_maxacc.items():
    for seed, epochs_at_maxacc in seed_to_epoch_at_maxacc.items():
        mean_epoch_at_maxacc = np.mean([int(e) for e in epochs_at_maxacc])
        std_epoch_at_maxacc = np.std([int(e) for e in epochs_at_maxacc])
        print(f"InitVecType: {initVecType:15}, Seed: {seed}, Mean Epoch number at Max Accuracy: {mean_epoch_at_maxacc:.2f}, Std: {std_epoch_at_maxacc:.2f}")


InitVecType: category_COG   , Seed: 0, Mean Max Accuracy: 0.4599, Std: 0.1151
InitVecType: category_COG   , Seed: 0, Mean Epoch number at Max Accuracy: 1.80, Std: 1.29


In [23]:
for initVecType, seed_to_maxacc in initVecType_to_seed_to_maxacc.items():
    for seed, max_accs in seed_to_maxacc.items():
        print(f"InitVecType: {initVecType:15}, Seed: {seed}, Max Accuracies: {[round(acc, 2) for acc in max_accs]}")

InitVecType: category_COG   , Seed: 0, Max Accuracies: [0.35, 0.23, 0.58, 0.43, 0.58, 0.22, 0.24, 0.3, 0.42, 0.52, 0.23, 0.36, 0.5, 0.34, 0.35, 0.54, 0.37, 0.51, 0.54, 0.52, 0.51, 0.5, 0.54, 0.34, 0.36, 0.56, 0.31, 0.33, 0.44, 0.39, 0.68, 0.52, 0.43, 0.74, 0.32, 0.59, 0.48, 0.48, 0.78, 0.4, 0.39, 0.77, 0.3, 0.5, 0.34, 0.56, 0.44, 0.79, 0.31, 0.54, 0.33, 0.53, 0.68, 0.44, 0.45, 0.46, 0.51, 0.41, 0.37, 0.24, 0.37, 0.46, 0.71, 0.47, 0.55, 0.62, 0.47, 0.38, 0.66, 0.27, 0.36, 0.4, 0.55, 0.31, 0.45, 0.39, 0.44, 0.43, 0.36, 0.29, 0.4, 0.33, 0.75, 0.35, 0.49, 0.22, 0.56, 0.54, 0.4, 0.4, 0.53, 0.45, 0.52, 0.48, 0.36, 0.49, 0.25, 0.53, 0.33, 0.48]
InitVecType: other_category_COG, Seed: 0, Max Accuracies: [0.37, 0.26, 0.51, 0.4, 0.56, 0.41, 0.21, 0.27, 0.58, 0.6, 0.21, 0.33, 0.52, 0.34, 0.53, 0.41, 0.42, 0.41, 0.72, 0.67, 0.44, 0.43, 0.42, 0.34, 0.4, 0.59, 0.31, 0.28, 0.66, 0.3, 0.45, 0.48, 0.43, 0.67, 0.33, 0.56, 0.4, 0.43, 0.89, 0.51, 0.36, 0.57, 0.39, 0.47, 0.34, 0.33, 0.53, 0.61, 0.31, 0.62, 

### とりあえず全部線グラフにする

In [ ]:
epoch = 10
initVecType_to_epoch_to_accs = defaultdict(lambda: defaultdict(list))
for initVecType, seed_to_result in initVecType_to_results.items():
    for seed, result in seed_to_result.items():
        for 
        for epoch, scores in result.items():
            print(epoch, scores)
            # initVecType_to_epoch_to_accs[initVecType][epoch].append(scores['accuracy']) 
# initVecType_to_epoch_to_accs